# Café GPU evaluation — S1, S2, S4

Use a fresh private Kaggle T4 notebook with Internet and the HF_TOKEN secret.
Fill in the same exact Git commit and the merged revision from training's final cell.
Each system runs gold-history turns and complete real-agent conversations against
fresh backends on synthetic test (400) and independent_test (84 Claude-authored cases
with Codex gold). Worked examples are excluded. No Groq requests occur here.

S1 and S4 use the exact shared prompt; S2 inserts four frozen train examples.
Temperature 0, max_tokens 512, seed 42, float16, context 8,192, single T4,
gpu_memory_utilization=0.85. vLLM V0/XFormers supports the T4 configuration.
Installation/load failure triggers a fresh Transformers process; generation errors
stop rather than mixing backends in one prediction file. Actual backends/revisions
and provenance are recorded in manifests. These GPU paths remain unverified until run.


In [ ]:
GITHUB_USERNAME = ""  # Your GitHub username; no token here.
REPO_NAME = "cafe-tool-agent"  # Match the repository you push.
GIT_REF = "PASTE_PHASE4_COMMIT"  # Exact 40-character project commit.
PRIVATE_GITHUB = False  # True requires an attached GITHUB_TOKEN Kaggle secret.
HF_USERNAME = ""  # Blank: derive the account name from HF_TOKEN.

IS_TRAINING = False
MERGED_REVISION = "6023f33114834618276c7de0cf43fda45397e1d7"
RESUME_PREDICTIONS = ""  # Optional prior /kaggle/input/... directory of backend folders.

In [ ]:
import base64
import json
import os
import re
import subprocess
import sys
from pathlib import Path

from kaggle_secrets import UserSecretsClient

if not re.fullmatch(r"[a-f0-9]{40}", GIT_REF):
    raise ValueError("Set GIT_REF to the Phase 4 commit from Codex's report before running")
if not re.fullmatch(r"[A-Za-z0-9-]+", GITHUB_USERNAME):
    raise ValueError("Fill in GITHUB_USERNAME")
if not re.fullmatch(r"[A-Za-z0-9_.-]+", REPO_NAME):
    raise ValueError("Invalid repository name")
secrets = UserSecretsClient()
ENV = os.environ.copy()
ENV["HF_TOKEN"] = secrets.get_secret("HF_TOKEN")
if IS_TRAINING:
    ENV["WANDB_API_KEY"] = secrets.get_secret("WANDB_API_KEY")
if PRIVATE_GITHUB:
    github_token = secrets.get_secret("GITHUB_TOKEN")
    credential = base64.b64encode(("x-access-token:" + github_token).encode()).decode()
    ENV.update(
        GIT_CONFIG_COUNT="1",
        GIT_CONFIG_KEY_0="http.https://github.com/.extraheader",
        GIT_CONFIG_VALUE_0="AUTHORIZATION: basic " + credential,
    )
    del github_token, credential
ENV.update(
    CUDA_VISIBLE_DEVICES="0",
    TOKENIZERS_PARALLELISM="false",
    PYTHONHASHSEED="42",
    HF_HOME="/tmp/cafe-hf",
    VLLM_USE_V1="0",
    VLLM_ATTENTION_BACKEND="XFORMERS",
    CUBLAS_WORKSPACE_CONFIG=":4096:8",
)
# No credential appears in a command, URL, saved manifest or printed environment.
subprocess.run(
    [sys.executable, "-m", "pip", "install", "--quiet", "--no-deps", "uv==0.11.26"], check=True
)
UV = [sys.executable, "-m", "uv"]
REQUIREMENTS = [
    "torch==2.6.0",
    "transformers==4.51.3",
    "accelerate==1.6.0",
    "peft==0.15.2",
    "trl==0.17.0",
    "bitsandbytes==0.45.5",
    "datasets==3.5.0",
    "wandb==0.22.3",
    "safetensors==0.5.3",
    "numpy==1.26.4",
    "sentencepiece==0.2.0",
    "protobuf==5.29.4",
    "tiktoken==0.9.0",
    "tokenizers==0.21.4",
    "opencv-python-headless==4.11.0.86",
]
PIN_FILE = Path("/tmp/cafe-notebook-requirements.txt")
PIN_FILE.write_text("\n".join(REQUIREMENTS) + "\n")
PACKAGE = f"git+https://github.com/{GITHUB_USERNAME}/{REPO_NAME}.git@{GIT_REF}"


def make_environment(location):
    python = str(Path(location) / "bin/python")
    if not Path(python).is_file():
        subprocess.run(UV + ["venv", "--python", "3.11", str(location)], check=True, env=ENV)
    subprocess.run(
        UV
        + [
            "pip",
            "install",
            "--python",
            python,
            "--exclude-newer",
            "2026-09-30T00:00:00Z",
            "-r",
            str(PIN_FILE),
            PACKAGE,
        ],
        check=True,
        env=ENV,
    )
    return python


PYTHON = make_environment("/tmp/cafe-notebook-env")
if not HF_USERNAME:
    HF_USERNAME = subprocess.check_output(
        [
            PYTHON,
            "-c",
            "import os; from huggingface_hub import HfApi; "
            "print(HfApi(token=os.environ['HF_TOKEN']).whoami()['name'])",
        ],
        env=ENV,
        text=True,
    ).strip()
DATASET_REPO = HF_USERNAME + "/cafe-tool-calls"
print("Python 3.11 environment ready; credentials remain in process memory.")

In [ ]:
result = subprocess.run(
    UV
    + [
        "pip",
        "install",
        "--python",
        PYTHON,
        "--exclude-newer",
        "2026-09-30T00:00:00Z",
        "--constraint",
        str(PIN_FILE),
        "vllm==0.8.5.post1",
    ],
    env=ENV,
)
VLLM_READY = result.returncode == 0
FALLBACK_PYTHON = None
print(
    "vLLM install:", "ready" if VLLM_READY else "failed; fresh Transformers fallback will be used"
)

## Run both evaluation modes

All systems reuse the shared strict parser, tool executor, logical clock and
append-only runner. Each model/backend has an immutable manifest and separate files.
To resume an interrupted session, save output as a private Kaggle input and set
RESUME_PREDICTIONS to the previous predictions directory containing backend folders.
Preserve the same Git/model revisions and settings. Completed IDs are skipped.
No history is truncated; context overflow is recorded as a length failure.


In [ ]:
if not re.fullmatch(r"[a-f0-9]{40}", MERGED_REVISION):
    raise ValueError("Copy the exact merged commit from the training export summary")
OUTPUT = Path("/kaggle/working/cafe-gpu")
OUTPUT.mkdir(exist_ok=True)
if RESUME_PREDICTIONS:
    # Only use your own saved output from the exact same Git/model revisions.
    subprocess.run(
        [
            PYTHON,
            "-c",
            "import sys; from pathlib import Path; "
            "from cafe_agent.kaggle.checkpoint import restore_predictions; "
            "restore_predictions(Path(sys.argv[1]), Path(sys.argv[2]))",
            RESUME_PREDICTIONS,
            str(OUTPUT / "predictions"),
        ],
        check=True,
        env=ENV,
    )
selection, attempts = {}, []
for system in ["S1", "S2", "S4"]:
    backend = "vllm" if VLLM_READY else "transformers"
    while True:
        if backend == "transformers" and FALLBACK_PYTHON is None:
            # A failed vLLM load cannot retain CUDA state in this new environment/process.
            FALLBACK_PYTHON = make_environment("/tmp/cafe-transformers-env")
        python = PYTHON if backend == "vllm" else FALLBACK_PYTHON
        directory = OUTPUT / "predictions" / backend
        command = [
            python,
            "-m",
            "cafe_agent.kaggle.evaluate",
            "--system",
            system,
            "--backend",
            backend,
            "--dataset-repo",
            DATASET_REPO,
            "--merged-repo",
            HF_USERNAME + "/cafe-agent-qwen2.5-1.5b-lora-merged",
            "--merged-revision",
            MERGED_REVISION,
            "--output",
            str(directory),
        ]
        result = subprocess.run(command, env=ENV)
        attempts.append({"system": system, "backend": backend, "exit_code": result.returncode})
        (OUTPUT / "attempts.json").write_text(json.dumps(attempts, indent=2))
        if result.returncode == 42 and backend == "vllm":
            VLLM_READY = False
            backend = "transformers"
            continue
        result.check_returncode()
        selection[system] = str(directory)
        break
(OUTPUT / "selection.json").write_text(json.dumps(selection, indent=2))

## Download for local scoring

The final cell checks complete coverage before bundling S1/S2/S4 predictions and
manifests. Download the ZIP, extract into results/predictions/ alongside S3, and
give the files back to Codex for Phase 5's shared local report. Do not overwrite
the S3 files. If interrupted, keep the partial backend folders for a later resume.


In [ ]:
from IPython.display import FileLink, display

subprocess.run(
    [
        PYTHON,
        "-m",
        "cafe_agent.kaggle.bundle",
        "--selection",
        str(OUTPUT / "selection.json"),
        "--output",
        str(OUTPUT / "gpu_predictions.zip"),
    ],
    check=True,
    env=ENV,
)
display(FileLink(str(OUTPUT / "gpu_predictions.zip")))
print("In the saved version's Output, download cafe-gpu/gpu_predictions.zip.")
print("Extract it into local results/predictions/, retaining existing S3 files.")